VERİ HAZIRLIĞI VE MODEL EĞİTİM, TAHMİN

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, r2_score  # R2 Score eklendi
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
import warnings

warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# AYARLAR
# ---------------------------------------------------------
folder = r"../../../data/by_sector"
EPOCHS = 300           
PATIENCE = 30          
BATCH_SIZE = 16        
L2_RATE = 0.01         

results = []
future_predictions = [] 
company_2026_predictions = [] 

print(f"GLOBAL LOG-LSTM MODELİ (LEAKAGE FIX + METRICS)")
print(f"Ayarlar: Epochs={EPOCHS}, Patience={PATIENCE}, L2={L2_RATE}\n")

for file in os.listdir(folder):
    if not file.startswith("Sector_") or not file.endswith(".csv"):
        continue
        
    sector = file.replace("Sector_", "").replace(".csv", "").replace("_", " ")
    path = os.path.join(folder, file)
    
    print(f"\n{'='*60}")
    print(f"SEKTÖR: {sector.upper()}")
    print(f"{'='*60}")
    
    df = pd.read_csv(path)
    df = df.sort_values(['CompanyID', 'Year']).reset_index(drop=True)
    
    # --- ADIM 1: LOG TRANSFORM ---
    df['Revenue_Log'] = np.log1p(df['Revenue'])
    
    # --- ADIM 2: LAG ÖZELLİKLERİ ---
    features_to_lag = ['Revenue_Log', 'ProfitMargin', 'GrowthRate']
    for col in features_to_lag:
        df[f'{col}_lag1'] = df.groupby('CompanyID')[col].shift(1)
    
    df = df.dropna().reset_index(drop=True)
    
    # --- ÖZELLİK SEÇİMİ ---
    features = ['Revenue_Log', 'ProfitMargin', 'GrowthRate', 
                'Revenue_Log_lag1', 'ProfitMargin_lag1', 'GrowthRate_lag1']
    
    # --- ADIM 3: SEQUENCE OLUŞTURMA (3 YIL -> 1 YIL) ---
    X_sequences, y_sequences = [], []
    
    for cid in df['CompanyID'].unique():
        company_data = df[df['CompanyID'] == cid].reset_index(drop=True)
        if len(company_data) < 4: 
            continue
            
        data_val = company_data[features].values
        target_val = company_data[['Revenue_Log', 'ProfitMargin']].values
        
        for i in range(len(company_data) - 3):
            X_sequences.append(data_val[i:i+3])
            y_sequences.append(target_val[i+3])
            
    if len(X_sequences) == 0:
        print("Yetersiz veri.")
        continue
        
    X_seq = np.array(X_sequences)
    y_seq = np.array(y_sequences)
    
    # --- ADIM 4: ROBUST SCALING (LEAKAGE FIXED) ---
    split = int(0.85 * len(X_seq))
    
    X_train_raw, X_test_raw = X_seq[:split], X_seq[split:]
    y_train_raw, y_test_raw = y_seq[:split], y_seq[split:]
    
    scaler_X = RobustScaler()
    scaler_y = RobustScaler()
    
    N_train, T, F = X_train_raw.shape
    N_test, _, _ = X_test_raw.shape
    
    # Sadece TRAIN fit edilir
    X_train_reshaped = X_train_raw.reshape(-1, F)
    X_train = scaler_X.fit_transform(X_train_reshaped).reshape(N_train, T, F)
    y_train = scaler_y.fit_transform(y_train_raw)
    
    # TEST transform edilir
    X_test_reshaped = X_test_raw.reshape(-1, F)
    X_test = scaler_X.transform(X_test_reshaped).reshape(N_test, T, F)
    y_test = scaler_y.transform(y_test_raw)
    
    # --- ADIM 5: MODEL EĞİTİMİ ---
    model = Sequential([
        LSTM(64, input_shape=(T, F), return_sequences=False, kernel_regularizer=l2(L2_RATE)),
        Dropout(0.4), 
        Dense(32, activation='relu', kernel_regularizer=l2(L2_RATE)),
        Dense(2)
    ])
    
    opt = Adam(learning_rate=0.001)
    model.compile(optimizer=opt, loss='mse')
    
    es = EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True)
    
    print(f"--> Model eğitiliyor ({len(X_train)} örnek)...")
    history = model.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, 
              validation_split=0.1, callbacks=[es], verbose=1)
    
    print(f"    Eğitim tamamlandı. Final Loss: {history.history['loss'][-1]:.5f}")

    # --- TEST PERFORMANSI ---
    pred_test_scaled = model.predict(X_test, verbose=0)
    pred_test = scaler_y.inverse_transform(pred_test_scaled)
    y_test_real = scaler_y.inverse_transform(y_test)
    
    # Log -> Gerçek Para
    pred_revenue_real = np.expm1(pred_test[:, 0]) 
    true_revenue_real = np.expm1(y_test_real[:, 0])
    
    # METRİKLER (Buraya Eklendi)
    mae_rev = mean_absolute_error(true_revenue_real, pred_revenue_real)
    mae_prof = mean_absolute_error(y_test_real[:, 1], pred_test[:, 1])
    
    # MAPE Hesaplama (Sıfıra bölünme hatasını önlemek için +1e-10 eklendi)
    mape_rev = np.mean(np.abs((true_revenue_real - pred_revenue_real) / (true_revenue_real + 1e-10))) * 100
    
    # R2 Score Hesaplama
    r2_rev = r2_score(true_revenue_real, pred_revenue_real)
    
    print(f"    Test MAE (Revenue):  {mae_rev:,.0f}")
    print(f"    Test MAPE (Revenue): %{mape_rev:.2f}")  # EKLENDİ
    print(f"    Test R2   (Revenue): {r2_rev:.3f}")     # EKLENDİ
    print(f"    Test MAE (Margin):   {mae_prof:.2f}")
    
    results.append({
        'Sector': sector,
        'Companies': df['CompanyID'].nunique(),
        'Sequences': len(X_sequences),
        'MAE_Revenue': round(mae_rev, 0),
        'MAPE_Revenue': round(mape_rev, 2), # EKLENDİ
        'R2_Score': round(r2_rev, 3),       # EKLENDİ
        'MAE_Margin': round(mae_prof, 2)
    })

    # --- ADIM 6: 2026 TAHMİNİ ---
    print("--> 2026 Tahminleri hesaplanıyor...")
    last_windows = []
    company_ids_ordered = []

    for cid in df['CompanyID'].unique():
        c_data = df[df['CompanyID'] == cid].sort_values('Year')
        if len(c_data) >= 3 and c_data.iloc[-1]['Year'] == 2025:
            last_3 = c_data.iloc[-3:][features].values
            last_windows.append(last_3)
            company_ids_ordered.append(int(cid))
            
    if len(last_windows) > 0:
        last_X = np.array(last_windows)
        N_last, T_last, F_last = last_X.shape
        
        last_X_reshaped = last_X.reshape(-1, F_last)
        last_X_scaled = scaler_X.transform(last_X_reshaped).reshape(N_last, T_last, F_last)
        
        pred_2026_scaled = model.predict(last_X_scaled, verbose=0)
        pred_2026 = scaler_y.inverse_transform(pred_2026_scaled)
        
        for i, cid in enumerate(company_ids_ordered):
            pred_rev_2026 = np.expm1(pred_2026[i, 0])
            pred_margin_2026 = pred_2026[i, 1]
            
            row_2025 = df[(df['CompanyID'] == cid) & (df['Year'] == 2025)]
            if not row_2025.empty:
                last_revenue_2025 = row_2025.iloc[0]['Revenue']
                growth_percent = (pred_rev_2026 / last_revenue_2025 - 1) * 100
                
                company_2026_predictions.append({
                    'CompanyID': cid,
                    'Sector': sector,
                    '2025_Revenue': round(last_revenue_2025, 1),
                    '2026_Predicted_Revenue': round(pred_rev_2026, 1),
                    '2026_Growth_%': round(growth_percent, 2),
                    '2026_Predicted_ProfitMargin_%': round(pred_margin_2026, 2)
                })
        
        avg_rev_2026 = np.mean(np.expm1(pred_2026[:, 0]))
        avg_prof_2026 = np.mean(pred_2026[:, 1])
        
        future_predictions.append({
            'Sector': sector,
            '2026_Avg_Revenue': round(avg_rev_2026, 0),
            '2026_Avg_Margin': round(avg_prof_2026, 2)
        })
        
        print(f"    Sektör Ort. Gelir: {avg_rev_2026:,.0f} | Şirket Sayısı: {len(company_ids_ordered)}")

# =========================
# KAYIT
# =========================
res_df = pd.DataFrame(results)
fut_df = pd.DataFrame(future_predictions)
company_pred_df = pd.DataFrame(company_2026_predictions)

print("\n" + "="*90)
print("TÜM İŞLEMLER TAMAMLANDI")
print("="*90)
display(res_df)
display(fut_df)

res_df.to_csv("Sector_Performance_Summary_Metrics.csv", index=False)
fut_df.to_csv("2026_Sector_Averages_Metrics.csv", index=False)
company_pred_df.to_csv("2026_All_Companies_Predictions_Metrics.csv", index=False)

print(f"\nDosyalar Kaydedildi (Versiyon: Metrics Added):")
print(f"1. Sector_Performance_Summary_Metrics.csv")
print(f"2. 2026_Sector_Averages_Metrics.csv")
print(f"3. 2026_All_Companies_Predictions_Metrics.csv")

GLOBAL LOG-LSTM MODELİ (LEAKAGE FIX + METRICS)
Ayarlar: Epochs=300, Patience=30, L2=0.01


SEKTÖR: CONSUMER GOODS
--> Model eğitiliyor (690 örnek)...
Epoch 1/300
39/39 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step - loss: 0.8755 - val_loss: 0.5490
Epoch 2/300
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.5076 - val_loss: 0.3977
Epoch 3/300
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.3819 - val_loss: 0.3100
Epoch 4/300
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.3066 - val_loss: 0.2465
Epoch 5/300
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 0.2491 - val_loss: 0.2034
Epoch 6/300
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.2136 - val_loss: 0.1753
Epoch 7/300
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.1819 - val_loss: 0.1493
Epoch 8/300
39/39 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.1587 - val_loss: 0.1326
Epoch 9/300
36/39 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.1478